# EduPredict: Deep Neural Network (DNN) Training in Google Colab

This notebook trains and evaluates the Deep Learning model for the **EduPredict** student academic performance prediction platform using TensorFlow/Keras.

---

### 🎯 Project Configuration & Rules
- **Target:** `G3` mapped to four performance categories:
  - `0`: Poor (`0–9`)
  - `1`: Average (`10–12`)
  - `2`: Good (`13–15`)
  - `3`: Excellent (`16–20`)
- **Input Features:** 58 transformed features (StandardScaler applied to numerical attributes including `G1` and `G2`, OneHotEncoder applied to categorical attributes).
- **Consistency Guarantee:** Preprocessing and stratified train/val/test splits were generated during Phase 2. **Do NOT refit the preprocessor or regenerate the dataset split in Colab.** Both the Random Forest and DNN are evaluated on the exact same test partition.

---

### 📋 Agreed DNN Architecture
- `Input(shape=(58,))`
- `Dense(64, activation='relu')`
- `Dropout(0.25)`
- `Dense(32, activation='relu')`
- `Dropout(0.15)`
- `Dense(4, activation='softmax')`

**Training Settings:**
- Optimizer: `Adam(learning_rate=0.001)`
- Loss: `sparse_categorical_crossentropy`
- Batch Size: 32
- Max Epochs: 100
- EarlyStopping: `monitor='val_loss', patience=12, restore_best_weights=True`

## 1. Verify Environment & Import Dependencies

In [ ]:
import os
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

# Set reproducible random seeds
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {bool(tf.config.list_physical_devices('GPU'))}")

## 2. Upload Phase 2 Preprocessed Artifacts

Upload the following files generated in Phase 2 from the local project's `artifacts/` folder:
1. `X_train.npy`
2. `y_train.npy`
3. `X_val.npy`
4. `y_val.npy`
5. `X_test.npy`
6. `y_test.npy`
7. `feature_names.json`
8. `class_mapping.json`

In [ ]:
try:
    from google.colab import files
    print("Please upload the 8 artifact files from your local 'artifacts/' directory:")
    uploaded = files.upload()
except ImportError:
    print("Running outside Google Colab. Expecting artifacts in local 'artifacts/' directory.")

## 3. Load & Verify Artifact Integrity

In [ ]:
# Determine search directory (current working directory or artifacts/)
artifact_dir = "artifacts" if os.path.exists("artifacts") else "."

X_train = np.load(os.path.join(artifact_dir, "X_train.npy"))
y_train = np.load(os.path.join(artifact_dir, "y_train.npy"))
X_val = np.load(os.path.join(artifact_dir, "X_val.npy"))
y_val = np.load(os.path.join(artifact_dir, "y_val.npy"))
X_test = np.load(os.path.join(artifact_dir, "X_test.npy"))
y_test = np.load(os.path.join(artifact_dir, "y_test.npy"))

with open(os.path.join(artifact_dir, "feature_names.json"), "r", encoding="utf-8") as f:
    feature_names = json.load(f)
with open(os.path.join(artifact_dir, "class_mapping.json"), "r", encoding="utf-8") as f:
    class_mapping = json.load(f)

target_names = [class_mapping[str(i)]['name'] for i in range(len(class_mapping))]

print("=" * 65)
print("DATASET VERIFICATION CHECKS:")
print(f"  - X_train shape: {X_train.shape}, dtype: {X_train.dtype}")
print(f"  - y_train shape: {y_train.shape}, dtype: {y_train.dtype}")
print(f"  - X_val shape:   {X_val.shape}, dtype: {X_val.dtype}")
print(f"  - y_val shape:   {y_val.shape}, dtype: {y_val.dtype}")
print(f"  - X_test shape:  {X_test.shape}, dtype: {X_test.dtype}")
print(f"  - y_test shape:  {y_test.shape}, dtype: {y_test.dtype}")
print(f"  - Total transformed features: {len(feature_names)}")
print(f"  - Target performance classes: {target_names}")

# Integrity Assertions
assert X_train.shape[1] == 58, f"Expected 58 input features, got {X_train.shape[1]}"
assert not np.isnan(X_train).any(), "NaNs detected in X_train"
assert not np.isnan(X_test).any(), "NaNs detected in X_test"
print("Zero NaNs detected across all arrays. Split integrity fully verified!")
print("=" * 65)

## 4. Build the Deep Neural Network Architecture

In [ ]:
def create_dnn_model(input_dim=58, num_classes=4, learning_rate=0.001):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(64, activation="relu", name="dense_1"),
        layers.Dropout(0.25, name="dropout_1"),
        layers.Dense(32, activation="relu", name="dense_2"),
        layers.Dropout(0.15, name="dropout_2"),
        layers.Dense(num_classes, activation="softmax", name="output_layer")
    ], name="EduPredict_DNN")
    
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

model = create_dnn_model(input_dim=X_train.shape[1], num_classes=len(class_mapping))
model.summary()

## 5. Train the Model with EarlyStopping

In [ ]:
early_stopping = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=12,
    restore_best_weights=True,
    verbose=1
)

EPOCHS = 100
BATCH_SIZE = 32

print("Commencing DNN training...")
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping],
    verbose=1
)
print("Training completed successfully.")

## 6. Plot Learning Curves (Loss & Accuracy)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss Curve
axes[0].plot(history.history['loss'], label='Training Loss', color='#2980b9', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', color='#e74c3c', linewidth=2, linestyle='--')
axes[0].set_title('DNN Training & Validation Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Sparse Categorical Crossentropy', fontsize=11)
axes[0].legend(frameon=True)

# Accuracy Curve
axes[1].plot(history.history['accuracy'], label='Training Accuracy', color='#27ae60', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', color='#f39c12', linewidth=2, linestyle='--')
axes[1].set_title('DNN Training & Validation Accuracy', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Accuracy', fontsize=11)
axes[1].legend(frameon=True)

plt.tight_layout()
plt.savefig('dnn_training_curves.png', dpi=300)
plt.show()

## 7. Evaluate on the Untouched Test Set

In [ ]:
# Compute probabilities and class predictions
test_probs = model.predict(X_test)
test_preds = np.argmax(test_probs, axis=1)

# Compute Metrics
acc = accuracy_score(y_test, test_preds)
macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(y_test, test_preds, average='macro', zero_division=0)
weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(y_test, test_preds, average='weighted', zero_division=0)
per_class_p, per_class_r, per_class_f1, per_class_sup = precision_recall_fscore_support(y_test, test_preds, average=None, zero_division=0)

clf_report_str = classification_report(y_test, test_preds, target_names=target_names, zero_division=0)
clf_report_dict = classification_report(y_test, test_preds, target_names=target_names, output_dict=True, zero_division=0)
cm = confusion_matrix(y_test, test_preds)

print("=" * 65)
print("DEEP NEURAL NETWORK: FINAL TEST EVALUATION METRICS")
print("=" * 65)
print(f"  Accuracy:        {acc:.4f} ({acc:.2%})")
print(f"  Macro Precision: {macro_p:.4f}")
print(f"  Macro Recall:    {macro_r:.4f}")
print(f"  Macro F1-Score:  {macro_f1:.4f}  <-- Primary Evaluation Metric")
print(f"  Weighted F1:     {weighted_f1:.4f}\n")

print("Per-Class Metrics on Test Set:")
for idx, name in enumerate(target_names):
    print(f"  Class {idx} ({name:<9}): Precision = {per_class_p[idx]:.4f}, Recall = {per_class_r[idx]:.4f}, F1 = {per_class_f1[idx]:.4f}, Support = {int(per_class_sup[idx])}")

print("\nClassification Report:")
print(clf_report_str)

# Confusion Matrix Plot
plt.figure(figsize=(7, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Greens",
    xticklabels=target_names, yticklabels=target_names,
    linewidths=0.8, linecolor="#eeeeee"
)
plt.title("Deep Neural Network: Test Confusion Matrix", fontsize=13, fontweight='bold', pad=12)
plt.xlabel("Predicted Label", fontsize=11, fontweight='bold')
plt.ylabel("True Label", fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('dnn_confusion_matrix.png', dpi=300)
plt.show()

## 8. Export Outputs for Local Integration

The following files will be saved and bundled into `edupredict_dnn_outputs.zip`:
1. `dnn_model.keras` — Trained Keras model
2. `training_history.json` — Epoch loss and accuracy logs
3. `dnn_metrics.json` — Test evaluation metrics dictionary
4. `dnn_predictions.npy` — Test prediction array
5. `dnn_confusion_matrix.png` — Test confusion matrix chart

In [ ]:
# 1. Save Keras Model
model.save('dnn_model.keras')
print("Saved dnn_model.keras")

# 2. Save Test Predictions
np.save('dnn_predictions.npy', test_preds)
print("Saved dnn_predictions.npy")

# 3. Save Training History JSON
history_data = {
    "loss": [float(x) for x in history.history['loss']],
    "val_loss": [float(x) for x in history.history['val_loss']],
    "accuracy": [float(x) for x in history.history['accuracy']],
    "val_accuracy": [float(x) for x in history.history['val_accuracy']],
}
with open('training_history.json', 'w', encoding='utf-8') as f:
    json.dump(history_data, f, indent=2)
print("Saved training_history.json")

# 4. Save Metrics JSON
metrics_dict = {
    "model_name": "DeepNeuralNetwork",
    "architecture": {
        "layers": ["Dense(64, relu)", "Dropout(0.25)", "Dense(32, relu)", "Dropout(0.15)", "Dense(4, softmax)"],
        "optimizer": "Adam(lr=0.001)",
        "loss": "sparse_categorical_crossentropy"
    },
    "test_metrics": {
        "accuracy": float(acc),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_p),
        "weighted_recall": float(weighted_r),
        "weighted_f1": float(weighted_f1),
        "per_class": {
            name: {
                "class_index": idx,
                "precision": float(per_class_p[idx]),
                "recall": float(per_class_r[idx]),
                "f1_score": float(per_class_f1[idx]),
                "support": int(per_class_sup[idx]),
            }
            for idx, name in enumerate(target_names)
        }
    },
    "confusion_matrix": cm.tolist()
}
with open('dnn_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_dict, f, indent=2)
print("Saved dnn_metrics.json")

# Bundle outputs into a single zip archive for easy download
files_to_bundle = [
    'dnn_model.keras',
    'training_history.json',
    'dnn_metrics.json',
    'dnn_predictions.npy',
    'dnn_confusion_matrix.png'
]

zip_filename = 'edupredict_dnn_outputs.zip'
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for fname in files_to_bundle:
        if os.path.exists(fname):
            zipf.write(fname)
print(f"\nAll DNN artifacts bundled into: {zip_filename}")

## 9. Download Artifacts to Local Project

Run the cell below in Google Colab to download `edupredict_dnn_outputs.zip`.

**Local placement instructions:**
- Extract `dnn_model.keras` into your local `models/` directory.
- Extract `training_history.json`, `dnn_metrics.json`, `dnn_predictions.npy`, and `dnn_confusion_matrix.png` into your local `assets/` directory.
- Run `python src/compare_models.py` in your local project to view the side-by-side performance comparison!

In [ ]:
try:
    from google.colab import files
    print("Triggering automatic download of the bundled outputs:")
    files.download('edupredict_dnn_outputs.zip')
except ImportError:
    print("Artifacts saved locally in current working directory.")